<a href="https://colab.research.google.com/github/pegumzs/PSP3/blob/main/PSP3_trab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 Online Shoppers — Análise de Dados e Machine Learning

**Base:** [`billcampos/online-shoppers`](https://www.kaggle.com/datasets/billcampos/online-shoppers) (Kaggle)

Cada linha é uma **sessão de navegação** em um e-commerce (12.330 sessões ao longo de 1 ano). O objetivo é prever se a sessão **terminou em compra** (`Revenue`) — um problema de **classificação binária**.

**Roteiro:** carga dos dados → análise exploratória → pré-processamento → modelo → avaliação → conclusões.

---
## 1. Setup

In [ ]:
!pip install -q kagglehub

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

AZUL, LARANJA = "#4C72B0", "#DD8452"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
print("Pronto ✅")

---
## 2. Carregando os dados

Se o Kaggle pedir autenticação, a célula tem um plano B que carrega a mesma base de um espelho público.

In [ ]:
URL_ESPELHO = ("https://raw.githubusercontent.com/sharmaroshan/"
               "Online-Shoppers-Purchasing-Intention/master/online_shoppers_intention.csv")

try:
    import kagglehub
    caminho = kagglehub.dataset_download("billcampos/online-shoppers")
    arquivo = sorted(glob.glob(os.path.join(caminho, "**", "*.csv"), recursive=True))[0]
    df = pd.read_csv(arquivo)
    print("Fonte: Kaggle ✅")
except Exception as e:
    print(f"⚠️ Kaggle indisponível ({type(e).__name__}), usando espelho público...")
    df = pd.read_csv(URL_ESPELHO)

df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
print(f"Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

---
## 3. Conhecendo os dados

As colunas descrevem o comportamento na sessão: quantas páginas de cada tipo o usuário viu e quanto tempo passou nelas (`Administrative`, `Informational`, `ProductRelated` e seus `_Duration`), métricas de engajamento (`BounceRates`, `ExitRates`, `PageValues`) e o contexto da visita (`Month`, `VisitorType`, `Weekend`, `Region`, `TrafficType`, `Browser`, `OperatingSystems`).

O alvo é **`Revenue`**: a sessão virou compra ou não.

> ⚠️ `OperatingSystems`, `Browser`, `Region` e `TrafficType` são números **apenas por codificação** — não têm ordem. Vamos tratá-los como categóricos.

In [ ]:
print("Valores ausentes:", df.isna().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print(f"Base após remover duplicatas: {len(df):,} sessões\n")

df.info()

In [ ]:
df.describe().T.round(2)

---
## 4. Análise exploratória

### 4.1 A variável alvo

In [ ]:
ALVO = "Revenue"

contagem = df[ALVO].value_counts().sort_index()

plt.figure(figsize=(6, 4))
barras = plt.bar(["Sem compra", "Compra"], contagem.values,
                 color=[AZUL, LARANJA], edgecolor="white")
plt.bar_label(barras, fmt="%d", padding=3, fontweight="bold")
plt.ylim(0, contagem.max() * 1.15)
plt.title("Distribuição da variável alvo", fontweight="bold")
plt.ylabel("nº de sessões")
plt.tight_layout()
plt.show()

print(f"Taxa de conversão: {df[ALVO].mean()*100:.2f}%")

🔴 **Classes desbalanceadas (~85% / 15%).** Um modelo que chuta "ninguém compra" já acerta 85% — por isso **acurácia não serve** aqui. Vamos olhar **ROC AUC, recall e F1**.

### 4.2 Conversão por mês e por tipo de visitante

In [ ]:
ordem_meses = ["Jan", "Feb", "Mar", "Apr", "May", "June",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))

conv_mes = (df.groupby("Month")[ALVO].mean() * 100)
conv_mes = conv_mes.reindex([m for m in ordem_meses if m in conv_mes.index])
b1 = eixos[0].bar(conv_mes.index, conv_mes.values, color=AZUL, edgecolor="white")
eixos[0].bar_label(b1, fmt="%.1f%%", padding=2, fontsize=8)
eixos[0].set_title("Conversão por mês", fontweight="bold")
eixos[0].set_ylabel("conversão (%)")

conv_vis = (df.groupby("VisitorType")[ALVO].mean() * 100).sort_values(ascending=False)
b2 = eixos[1].bar(conv_vis.index, conv_vis.values, color=AZUL, edgecolor="white")
eixos[1].bar_label(b2, fmt="%.1f%%", padding=2, fontsize=9)
eixos[1].set_title("Conversão por tipo de visitante", fontweight="bold")
eixos[1].set_ylabel("conversão (%)")

for eixo in eixos:
    eixo.axhline(df[ALVO].mean() * 100, color=LARANJA, ls="--", lw=2)
    eixo.set_ylim(0, 32)

plt.tight_layout()
plt.show()

📌 **Novembro dispara (~25%)** — efeito Black Friday — enquanto **fevereiro é o pior mês (~2%)**. E, de forma contraintuitiva, **novos visitantes convertem mais** que os recorrentes. A linha laranja é a média global.

### 4.3 Comportamento de quem compra × quem não compra

In [ ]:
destaques = ["PageValues", "ExitRates", "ProductRelated"]

fig, eixos = plt.subplots(1, 3, figsize=(13, 4))
for i, col in enumerate(destaques):
    sns.boxplot(data=df, x=ALVO, y=col, hue=ALVO, palette=[AZUL, LARANJA],
                legend=False, showfliers=False, ax=eixos[i])
    eixos[i].set_title(col, fontweight="bold")
    eixos[i].set_xticks([0, 1])
    eixos[i].set_xticklabels(["Sem compra", "Compra"])
    eixos[i].set_xlabel("")

plt.suptitle("Comportamento na sessão por desfecho", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

df.groupby(ALVO)[destaques].mean().round(2)

📌 Quem compra tem **`PageValues` muito maior**, **taxa de saída menor** e visita **mais páginas de produto**. São exatamente os sinais que o modelo vai aprender.

> Observação: `PageValues` é uma métrica do Google Analytics calculada *a partir* de transações — em produção ela pode não estar disponível no começo da sessão. Mantemos porque é o padrão nos estudos com essa base, mas vale ter isso em mente.

---
## 5. Pré-processamento

Montamos tudo dentro de um **`Pipeline`**, que garante que o escalonamento e o *one-hot encoding* sejam aprendidos só no treino — evitando **vazamento de dados**.

- Numéricas → `StandardScaler`
- Categóricas → `OneHotEncoder`
- Divisão 80/20 **estratificada**, para manter a proporção de compras nos dois conjuntos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

y = df[ALVO].astype(int)
X = df.drop(columns=[ALVO]).copy()

# booleanas viram 0/1
for c in X.columns:
    if pd.api.types.is_bool_dtype(X[c]):
        X[c] = X[c].astype(int)

CODIFICADAS = ["OperatingSystems", "Browser", "Region", "TrafficType"]
col_categoricas = [c for c in X.columns
                   if (not pd.api.types.is_numeric_dtype(X[c])) or c in CODIFICADAS]
col_numericas = [c for c in X.columns if c not in col_categoricas]

preprocessador = ColumnTransformer([
    ("num", StandardScaler(), col_numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), col_categoricas),
])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Categóricas:", col_categoricas)
print(f"\nTreino: {len(X_treino):,} sessões | Teste: {len(X_teste):,} sessões")

---
## 6. Machine Learning

Treinamos dois modelos: uma **regressão logística** (linear, simples) e um **Random Forest** (conjunto de árvores, capta relações não lineares). O `class_weight="balanced"` faz o algoritmo dar mais peso à classe minoritária.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

modelos = {
    "Regressão Logística": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=RANDOM_STATE),
}

linhas, treinados = [], {}
for nome, estimador in modelos.items():
    pipe = Pipeline([("prep", preprocessador), ("clf", estimador)])
    pipe.fit(X_treino, y_treino)
    treinados[nome] = pipe

    pred = pipe.predict(X_teste)
    proba = pipe.predict_proba(X_teste)[:, 1]
    linhas.append({"Modelo": nome,
                   "Acurácia": accuracy_score(y_teste, pred),
                   "Precisão": precision_score(y_teste, pred),
                   "Recall": recall_score(y_teste, pred),
                   "F1": f1_score(y_teste, pred),
                   "ROC AUC": roc_auc_score(y_teste, proba)})
    print(f"✔ {nome} treinado")

resultados = pd.DataFrame(linhas).set_index("Modelo").round(4)
resultados

**Como ler as métricas:**

- **Precisão** — dos previstos como compradores, quantos compraram de fato (evita desperdiçar cupom).
- **Recall** — dos compradores reais, quantos o modelo encontrou (evita perder oportunidade).
- **ROC AUC** — capacidade de ordenar corretamente compradores e não compradores. É a métrica principal aqui.

---
## 7. Avaliação do melhor modelo

In [ ]:
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay)

melhor_nome = resultados["ROC AUC"].idxmax()
melhor = treinados[melhor_nome]
print(f"🏆 Melhor modelo: {melhor_nome}\n")
print(classification_report(y_teste, melhor.predict(X_teste),
                            target_names=["Sem compra", "Compra"], digits=3))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_estimator(
    melhor, X_teste, y_teste, ax=eixos[0], cmap="Blues", colorbar=False,
    display_labels=["Sem compra", "Compra"], values_format="d")
eixos[0].set_title("Matriz de confusão", fontweight="bold")
eixos[0].set_xlabel("Previsto"); eixos[0].set_ylabel("Real")

RocCurveDisplay.from_estimator(melhor, X_teste, y_teste, ax=eixos[1],
                               color=AZUL, name=melhor_nome)
eixos[1].plot([0, 1], [0, 1], "k--", lw=1, label="aleatório")
eixos[1].set_title("Curva ROC", fontweight="bold")
eixos[1].legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

A matriz de confusão traduz o modelo em decisões de negócio: **falsos positivos** são cupons desperdiçados, **falsos negativos** são compradores que o modelo deixou passar.

### Quais variáveis mais pesam?

In [ ]:
nomes = melhor.named_steps["prep"].get_feature_names_out()
importancias = pd.Series(melhor.named_steps["clf"].feature_importances_, index=nomes)
top = importancias.sort_values().tail(10)

plt.figure(figsize=(8, 4.5))
plt.barh([n.split("__")[-1] for n in top.index], top.values,
         color=AZUL, edgecolor="white")
plt.title("Top 10 variáveis mais importantes", fontweight="bold")
plt.xlabel("importância")
plt.tight_layout()
plt.show()

📌 **`PageValues` domina** o poder preditivo, seguida pelas métricas de engajamento (`ExitRates`, `ProductRelated`) e pela sazonalidade.

---
## 8. Conclusões

**Sobre os dados**
- Base limpa (sem valores ausentes) e **desbalanceada**: só ~15,6% das sessões viram compra.
- **Sazonalidade forte**: novembro converte ~25%, fevereiro ~2%.
- Quem compra navega mais páginas de produto e tem taxa de saída menor.

**Sobre o modelo**
- O **Random Forest** atingiu **ROC AUC ≈ 0,93** no conjunto de teste.
- Ou seja: o modelo ordena bem as sessões por propensão à compra — o suficiente para priorizar ações de marketing em tempo real.
- `PageValues` é de longe a variável mais importante.

**Próximos passos possíveis**
- Otimizar hiperparâmetros com `GridSearchCV`.
- Testar XGBoost ou LightGBM.
- Ajustar o limiar de decisão (o padrão 0,5 raramente é o ideal em bases desbalanceadas).

---
*Dados: UCI / Kaggle · Sakar, C.O. et al., Neural Computing and Applications, 2018.*